# 01 - POPE feature extraction with Qwen2-VL-2B (Google Colab)

**GPU REQUIRED - RUN IN COLAB - NOT TESTED LOCALLY** (the model-loading and inference cells have never been run on a GPU by the author of this notebook).

Do not run this notebook on your laptop. It is the canonical way to run the model; `src/feature_extraction.py` is not the entry point.

## Colab setup guide (same steps as `COLAB.md`)

1. **Get the notebook into Colab.** Commit and push *only the notebooks / COLAB.md* (never `data/`, `.venv/` or a full `.pt` cache) to GitHub, then in Colab use *File > Open notebook > GitHub* and pick `notebooks/01_colab_pope_feature_extraction.ipynb`. Alternatively upload the `.ipynb` with *File > Upload notebook*.
2. **Runtime.** *Runtime > Change runtime type > Python 3 > Hardware accelerator: GPU (T4 or better; L4/A100 are faster).* The 2B model needs about 6-8 GB of GPU memory at batch size 1.
3. **Run cells top to bottom**, in this order: `Config` -> `a. Colab setup` (installs, repo clone, optional Drive mount) -> `b. Dataset inspection` -> `c. Token check` -> `d. Prompt freeze` -> `e. Model load` -> `f. Scoring function` -> `g/h. Run extraction` -> `Merge` -> `i. Verification` -> (optional) `Download`.
4. **Outputs** (in `OUT_DIR`, default `results/cached_features/` inside the cloned repo, or your Drive folder if `USE_DRIVE=True`): `pope_full_features.pt` (about 0.8 GB for 9,000 questions), `pope_metadata.parquet` (a few MB), `pope_run_config.json`, and the shard folder `pope_full_shards/` (same data as the .pt, kept for resuming).
5. **Get outputs back.** Preferred: set `USE_DRIVE=True` in the config cell and copy the three files from Drive into your repo's `results/cached_features/`. Fallback: set `DOWNLOAD_AT_END=True` (uses `google.colab.files.download`; 0.8 GB downloads can be flaky in a browser, so prefer Drive).
6. **Verify.** The `i. Verification` cell must print: no missing rows, `pope_full_features.pt` size within the expected range, accuracy / false-yes rate per POPE setting, and no non-finite values.
7. **If Colab disconnects:** reconnect, re-run the cells from the top (same config). Finished 100-question shards are detected and skipped; at most the last 100 questions are redone. **If you hit CUDA OOM:** *Runtime > Restart session*, keep `BATCH_SIZE = 1`, re-run from the top.

Existing `results/cached_features/pope_subset_features.pt` is **never overwritten** by this notebook.

## Config (edit only this cell)

In [1]:
# ===================== CONFIG =====================
MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"       # same model as the existing cache; do not change without re-doing everything
DEVICE = "cuda"                               # this notebook is GPU-only
DTYPE = "bfloat16"                            # model compute dtype. NOTE: the existing 100-sample cache was made in float16 (see markdown below)
STORE_DTYPE = "float16"                       # dtype of saved hidden states (matches the existing cache; ~0.8 GB for 9,000 questions)

DATASET_NAME = "lmms-lab-encoder/POPE"
DATASET_CONFIG = None                         # None = HF default config (what src/download_datasets.py loads): 1 split 'test', 9000 rows, setting in field 'category'
DATASET_LOCAL_PATH = "data/pope"              # relative to repo root; loaded from disk if present, else downloaded and saved here

POPE_SETTINGS = None                          # None = all of random/popular/adversarial; or e.g. ["adversarial"]  (filters on the 'category' field)
SUBSET_SIZE = None                            # None = FULL RUN (default). An int takes the first N rows of the seeded processing order.
SHUFFLE_ORDER = True                          # process rows in a seeded random order, so a partial run is a random sample, not only 'adversarial'
SEED = 0
BATCH_SIZE = 1                                # keep 1 (see OOM note). >1 uses left padding and is less tested
CHECKPOINT_EVERY = 100                        # questions per saved shard

PROMPT_SUFFIX = " Answer yes or no."          # frozen: identical to src/feature_extraction.py -> f"{question} Answer yes or no."
SPLIT_FRACS = (0.70, 0.15, 0.15)              # train / dev / test, assigned BY IMAGE (hash of image id + SEED)

OUT_DIR = "results/cached_features"           # relative to repo root (ignored if USE_DRIVE)
FEATURES_NAME = "pope_full_features.pt"       # never the existing pope_subset_features.pt
META_NAME = "pope_metadata.parquet"           # falls back to .csv if parquet is unavailable
CONFIG_NAME = "pope_run_config.json"
SHARD_DIRNAME = "pope_full_shards"
KEEP_SHARDS = True

USE_DRIVE = False                             # True: mount Google Drive and write outputs to DRIVE_DIR
DRIVE_DIR = "/content/drive/MyDrive/sc4001_pope"
DOWNLOAD_AT_END = False                       # True: files.download() the final files at the end
REPO_URL = "https://github.com/RUANLASS/SC4001Assignment_2026.git"
# ==================================================

## a. Colab setup  —  GPU REQUIRED - RUN IN COLAB - NOT TESTED LOCALLY

In [2]:
# [GPU SETUP]  GPU REQUIRED - RUN IN COLAB - NOT TESTED LOCALLY
import os, sys, subprocess, json, hashlib, time, datetime, shutil
from pathlib import Path

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "nvidia-smi not found")

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "torch", "torchvision", "transformers>=4.45,<5", "accelerate", "datasets<5",
                       "scikit-learn", "pillow", "numpy", "pandas", "pyarrow", "matplotlib", "seaborn", "tqdm"])

import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > GPU, then re-run."
print("GPU:", torch.cuda.get_device_name(0), "| capability:", torch.cuda.get_device_capability(0),
      "| memory GB: %.1f" % (torch.cuda.get_device_properties(0).total_memory / 1e9))

IN_COLAB = "google.colab" in sys.modules

def find_repo_root(start=None):
    p = Path(start or os.getcwd()).resolve()
    for q in [p, *p.parents]:
        if (q / "src" / "download_datasets.py").exists():
            return q
    return None

ROOT = find_repo_root()
if ROOT is None:
    target = Path("/content/SC4001Assignment_2026") if IN_COLAB else Path.cwd() / "SC4001Assignment_2026"
    if not target.exists():
        r = subprocess.run(["git", "clone", REPO_URL, str(target)], capture_output=True, text=True)
        print(r.stdout, r.stderr)
    if (target / "src").exists():
        ROOT = target
    else:
        # Repo not reachable (private / notebook not pushed yet). This notebook does not import from src/, so continue in a scratch folder.
        ROOT = Path("/content/sc4001_run") if IN_COLAB else Path.cwd() / "sc4001_run"
        ROOT.mkdir(parents=True, exist_ok=True)
        print("WARNING: repo clone failed; using scratch folder", ROOT)
os.chdir(ROOT)
print("Working directory:", ROOT)

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = Path(DRIVE_DIR)
else:
    OUT = ROOT / OUT_DIR
OUT.mkdir(parents=True, exist_ok=True)
SHARD_DIR = OUT / SHARD_DIRNAME
SHARD_DIR.mkdir(exist_ok=True)
print("Outputs go to:", OUT)

Thu Nov  5 16:17:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 621.01                 KMD Version: 621.01        CUDA UMD Version: 13.5     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4070 ...  WDDM  |   00000000:01:00.0  On |                  N/A |
|  0%   44C    P5             32W /  285W |    2495MiB /  16376MiB |     19%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## b. Dataset inspection

Structure found during the repo audit (verified by downloading the dataset): the **default** config of `lmms-lab-encoder/POPE` has **one split, `test`, 9,000 rows**, with fields `id, question_id, question, answer, image_source, image, category`. The POPE setting (random / popular / adversarial) is **not** a split here; it is the `category` field (3,000 rows each, 1,500 yes / 1,500 no each). `question_id` restarts in each setting so it is **not unique**; `id` ("0".."8999") is. All 500 COCO images (`image_source`) appear in **all three** settings, which is why train/dev/test is split by image across settings. (The HF page also lists a separate `Full` config with 3 splits; this notebook uses the default config because that is what the existing cache was built from.)

The cell below re-prints all of this from the data actually loaded. It adapts to what it finds and refuses to continue if the structure is not the expected one.

In [3]:
import numpy as np, pandas as pd
from datasets import load_dataset, load_from_disk

local = ROOT / DATASET_LOCAL_PATH
if local.exists() and any(local.iterdir()):
    print("Loading POPE from disk:", local)
    ds = load_from_disk(str(local))
else:
    print("Downloading POPE from Hugging Face ...")
    ds = load_dataset(DATASET_NAME) if DATASET_CONFIG is None else load_dataset(DATASET_NAME, DATASET_CONFIG)
    local.parent.mkdir(parents=True, exist_ok=True)
    ds.save_to_disk(str(local))

print(ds)
print("Split names:", list(ds.keys()))
SPLIT_NAME = "test" if "test" in ds else (list(ds.keys())[0] if len(ds) == 1 else None)
assert SPLIT_NAME is not None, f"Unexpected split structure {list(ds.keys())}; this notebook handles the single-split default config only."
split_ds = ds[SPLIT_NAME]
FIELDS = list(split_ds.features.keys())
print("Using split:", SPLIT_NAME, "| fields:", FIELDS, "| rows:", len(split_ds))
for needed in ["question", "answer", "image", "image_source", "category"]:
    assert needed in FIELDS, f"Expected field '{needed}' missing; got {FIELDS}"

meta0 = split_ds.remove_columns("image").to_pandas()
meta0["row_idx"] = np.arange(len(meta0))
meta0["answer"] = meta0["answer"].str.strip().str.lower()
assert set(meta0["answer"]) <= {"yes", "no"}, set(meta0["answer"])
print("\nPOPE settings ('category'):", meta0["category"].value_counts().to_dict())
print("yes/no balance overall:", meta0["answer"].value_counts().to_dict())
print(pd.crosstab(meta0["category"], meta0["answer"]))
print("unique images:", meta0["image_source"].nunique(), "| images appearing in >1 setting:",
      int((meta0.groupby("image_source")["category"].nunique() > 1).sum()))
print("'id' unique:", meta0["id"].is_unique, "| 'question_id' unique:", meta0["question_id"].is_unique)
print("Row order of first 100 rows by setting:", meta0.head(100)["category"].value_counts().to_dict())

c:\Users\Toxic\SC4001\SC4001Assignment_2026\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


c:\Users\Toxic\SC4001\SC4001Assignment_2026\.venv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Toxic\.cache\huggingface\hub\datasets--lmms-lab-encoder--POPE. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Saving the dataset (3/3 shards): 100%|██████████| 9000/9000 [00:01<00:00, 5645.58 examples/s]

DatasetDict({
    test: Dataset({
        features: ['id', 'question_id', 'question', 'answer', 'image_source', 'image', 'category'],
        num_rows: 9000
    })
})
Split names: ['test']
Using split: test | fields: ['id', 'question_id', 'question', 'answer', 'image_source', 'image', 'category'] | rows: 9000

POPE settings ('category'): {'adversarial': 3000, 'popular': 3000, 'random': 3000}
yes/no balance overall: {'yes': 4500, 'no': 4500}
answer         no   yes
category               
adversarial  1500  1500
popular      1500  1500
random       1500  1500
unique images: 500 | images appearing in >1 setting: 500
'id' unique: True | 'question_id' unique: False
Row order of first 100 rows by setting: {'adversarial': 100}


## c. Token check: which tokens are scored as "yes" / "no"

The prompt ends with Qwen's assistant turn header (`<|im_start|>assistant\n`), so the model's first answer token is a word with **no leading space** ("Yes" / "No", sometimes lower case). Leading-space variants (" Yes") would only be natural mid-sentence. We therefore score with the **case variants without a leading space** that are single tokens: `Yes, yes, YES` vs `No, no, NO`. P(yes) is the sum of probabilities over the yes ids, P(no) likewise; the leading-space variants are printed below for inspection but **not** scored. We also save `answer_mass = P(yes)+P(no)` per question: if it is far below 1 the model is not answering in the requested format, and the saved `top1_token` shows what it wanted to say instead. The exact ids are frozen into `pope_run_config.json`, and a resumed run refuses to continue if they differ.

In [4]:
from transformers import AutoProcessor
processor = AutoProcessor.from_pretrained(MODEL_ID)
tok = processor.tokenizer

variants = ["Yes", "yes", "YES", " Yes", " yes", " YES", "No", "no", "NO", " No", " no", " NO"]
print(f"{'string':10s} {'token ids':20s} single-token?")
for v in variants:
    ids = tok.encode(v, add_special_tokens=False)
    print(f"{v!r:10s} {str(ids):20s} {len(ids) == 1}")

YES_STRINGS, NO_STRINGS = ["Yes", "yes", "YES"], ["No", "no", "NO"]
def single_ids(strings):
    out = {}
    for s in strings:
        ids = tok.encode(s, add_special_tokens=False)
        if len(ids) == 1:
            out[s] = ids[0]
        else:
            print(f"  dropping {s!r}: encodes to {len(ids)} tokens {ids}")
    return out
YES_MAP, NO_MAP = single_ids(YES_STRINGS), single_ids(NO_STRINGS)
YES_IDS, NO_IDS = sorted(set(YES_MAP.values())), sorted(set(NO_MAP.values()))
assert YES_IDS and NO_IDS and not set(YES_IDS) & set(NO_IDS)
print("\nFROZEN yes tokens:", YES_MAP, "\nFROZEN no tokens: ", NO_MAP)

c:\Users\Toxic\SC4001\SC4001Assignment_2026\.venv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Toxic\.cache\huggingface\hub\models--Qwen--Qwen2-VL-2B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by defa

string     token ids            single-token?
'Yes'      [9454]               True
'yes'      [9693]               True
'YES'      [14004]              True
' Yes'     [7414]               True
' yes'     [9834]               True
' YES'     [14080]              True
'No'       [2753]               True
'no'       [2152]               True
'NO'       [8996]               True
' No'      [2308]               True
' no'      [902]                True
' NO'      [5664]               True

FROZEN yes tokens: {'Yes': 9454, 'yes': 9693, 'YES': 14004} 
FROZEN no tokens:  {'No': 2753, 'no': 2152, 'NO': 8996}


## d. Prompt freeze

The prompt is **exactly** the one in `src/feature_extraction.py`: `f"{question} Answer yes or no."`, wrapped in the Qwen2-VL chat template with one image and `add_generation_prompt=True`. It is not changed here, so results stay comparable with the existing 100-sample cache.

Audit notes on the old script (flagged, **not** silently changed): (1) it loaded the model in **float16**, this notebook asks for **bfloat16**, so hidden states are not bit-comparable; set `DTYPE = "float16"` to match the old cache exactly (on a T4, which has no native bf16, float16 is also the faster and safer choice). (2) It contains an unused line `image_inputs, video_inputs = processor.image_processor(...)` that has no effect on the features. (3) `all_labels` was never filled and no predictions were saved; this notebook saves all of them.

In [5]:
PROMPT_TEMPLATE = "{question}" + PROMPT_SUFFIX
def make_prompt(question):
    return PROMPT_TEMPLATE.format(question=question)

def render_chat(question):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": make_prompt(question)}]}]
    return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print(repr(render_chat(meta0.loc[0, "question"])))

'<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\n<|vision_start|><|image_pad|><|vision_end|>Is there a snowboard in the image? Answer yes or no.<|im_end|>\n<|im_start|>assistant\n'


## e. Model load  —  GPU REQUIRED - RUN IN COLAB - NOT TESTED LOCALLY

bf16 on CUDA, eval mode, gradients disabled. **If you get a CUDA out-of-memory error: Runtime > Restart session, keep `BATCH_SIZE = 1`, and re-run from the top.**

In [6]:
# [GPU]  GPU REQUIRED - RUN IN COLAB - NOT TESTED LOCALLY
from transformers import Qwen2VLForConditionalGeneration
torch_dtype = getattr(torch, DTYPE)
if DTYPE == "bfloat16" and torch.cuda.get_device_capability(0)[0] < 8:
    print("WARNING: this GPU has no native bfloat16 (e.g. T4). It may be slow or fail; consider DTYPE='float16'.")

model = Qwen2VLForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=torch_dtype).to(DEVICE)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)
tok.padding_side = "left"          # so that index -1 is always the last prompt token, also when BATCH_SIZE > 1

cfg_text = model.config.get_text_config() if hasattr(model.config, "get_text_config") else model.config
N_LAYERS_TOTAL = cfg_text.num_hidden_layers + 1       # embeddings + each transformer layer
HIDDEN_SIZE = cfg_text.hidden_size
print(f"Loaded {MODEL_ID} in {DTYPE}. hidden_states entries per token: {N_LAYERS_TOTAL} (embeddings + {cfg_text.num_hidden_layers} layers), hidden size {HIDDEN_SIZE}")
print("Existing cache for comparison: 29 x 1536 (read from the file in notebook 00).")

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 31.74it/s]


Loaded Qwen/Qwen2-VL-2B-Instruct in bfloat16. hidden_states entries per token: 29 (embeddings + 28 layers), hidden size 1536
Existing cache for comparison: 29 x 1536 (read from the file in notebook 00).


## f. Forced-choice scoring (one forward pass per question)

For each question we run **one** forward pass with `output_hidden_states=True`, take the logits at the **last prompt token** (the position that predicts the first answer token), and compare the probability mass on the yes-ids vs the no-ids. No free generation is used. `prediction = "yes"` iff `P(yes) > P(no)`.

Saved per question: `p_yes`, `p_no` (raw full-vocab probabilities summed over the frozen ids), `p_yes_norm = P(yes)/(P(yes)+P(no))`, `margin = log P(yes) - log P(no)` (signed log-odds), `binary_entropy` (bits, of `p_yes_norm`), `vocab_entropy` (nats, full vocabulary; cheap because logits are already computed), `answer_mass`, `top1_token`.
Hidden states: `hidden_states[l][:, -1, :]` for every `l` (index 0 = embedding output, last index = final layer as returned by HF).

In [7]:
def score_logits(logits):
    # logits: [B, V] (any float dtype). Pure function; returns dict of CPU float tensors / lists.
    logp = torch.log_softmax(logits.float(), dim=-1)
    lp_yes = torch.logsumexp(logp[:, YES_IDS], dim=-1)
    lp_no = torch.logsumexp(logp[:, NO_IDS], dim=-1)
    margin = lp_yes - lp_no
    p_yes_norm = torch.sigmoid(margin)
    q = p_yes_norm.clamp(1e-12, 1 - 1e-12)
    bin_ent = -(q * torch.log2(q) + (1 - q) * torch.log2(1 - q))
    vocab_ent = -(logp.exp() * logp).sum(-1)
    top1 = logp.argmax(-1)
    return dict(p_yes=lp_yes.exp().cpu(), p_no=lp_no.exp().cpu(), p_yes_norm=p_yes_norm.cpu(), margin=margin.cpu(),
                binary_entropy=bin_ent.cpu(), vocab_entropy=vocab_ent.cpu(), top1_id=top1.cpu())

def forward_batch(questions, images):
    # [GPU]  Returns (hidden [B, L, H] float32 on CPU, last-token logits [B, V]).
    texts = [render_chat(q) for q in questions]
    inputs = processor(text=texts, images=images, padding=True, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model(**inputs, output_hidden_states=True, use_cache=False)
    hidden = torch.stack([h[:, -1, :] for h in out.hidden_states], dim=1).float().cpu()
    logits = out.logits[:, -1, :].float().cpu()
    return hidden, logits

def get_sample(row_idx):
    s = split_ds[int(row_idx)]
    return s["image"], s["question"]

## g/h. Run extraction with checkpointing and resume

Files are written **incrementally** as 100-question shards in `pope_full_shards/` (atomic write: temp file then rename), and finished rows are detected **by dataset row index**, so after a disconnect you just re-run from the top. The final `pope_full_features.pt` / `pope_metadata.parquet` are merged from the shards in the next cell.

Metadata `split` is assigned **by image** (`image_source`) with a hash, so all questions of an image, in all three settings, land in the same split.

In [8]:
def split_of(image_id):
    u = int(hashlib.md5(f"{SEED}-{image_id}".encode()).hexdigest()[:8], 16) / 0xFFFFFFFF
    return "train" if u < SPLIT_FRACS[0] else ("dev" if u < SPLIT_FRACS[0] + SPLIT_FRACS[1] else "test")

# ---- selection and processing order
cand = meta0 if POPE_SETTINGS is None else meta0[meta0["category"].isin(POPE_SETTINGS)]
assert len(cand) > 0, f"No rows for POPE_SETTINGS={POPE_SETTINGS}"
order = cand["row_idx"].to_numpy().copy()
if SHUFFLE_ORDER:
    np.random.default_rng(SEED).shuffle(order)
if SUBSET_SIZE is not None:
    order = order[:SUBSET_SIZE]
SELECTED = [int(r) for r in order]
print(f"Selected {len(SELECTED)} questions (full run = {len(meta0)}).")

# ---- run config (written now, completed at the end); a resumed run must match
RUN_CONFIG = dict(
    model_id=MODEL_ID, dtype=DTYPE, store_dtype=STORE_DTYPE,
    prompt_template=PROMPT_TEMPLATE, chat_rendering_example=render_chat("<QUESTION>"),
    yes_token_ids=YES_MAP, no_token_ids=NO_MAP,
    scoring="forced choice, logits at last prompt token, P(yes)=sum over yes ids, P(no)=sum over no ids; pred yes iff P(yes)>P(no)",
    token_position="last prompt token (index -1, left padding), all hidden_states entries",
    dataset=dict(name=DATASET_NAME, config=DATASET_CONFIG, local_path=DATASET_LOCAL_PATH, split=SPLIT_NAME,
                 fields=FIELDS, n_rows=len(meta0), settings_run=POPE_SETTINGS or sorted(meta0["category"].unique().tolist())),
    subset_size=SUBSET_SIZE, shuffle_order=SHUFFLE_ORDER, seed=SEED, batch_size=BATCH_SIZE,
    split_rule=f"md5('{{seed}}-{{image_source}}') -> train/dev/test fractions {SPLIT_FRACS}",
    n_hidden_state_entries=int(N_LAYERS_TOTAL), hidden_size=int(HIDDEN_SIZE),
    torch=torch.__version__, gpu=torch.cuda.get_device_name(0),
)
import transformers
RUN_CONFIG["transformers"] = transformers.__version__
CONFIG_PATH = OUT / CONFIG_NAME
CHECK_KEYS = ["model_id", "dtype", "prompt_template", "yes_token_ids", "no_token_ids", "seed", "shuffle_order", "token_position"]
if CONFIG_PATH.exists():
    old = json.loads(CONFIG_PATH.read_text())
    bad = [k for k in CHECK_KEYS if old.get(k) != RUN_CONFIG.get(k)]
    assert not bad, f"Existing run in {OUT} used different settings for {bad}. Use a new OUT_DIR or delete the old shards."
    print("Resuming a previous run (config matches).")
RUN_CONFIG.update(status="running", started_utc=datetime.datetime.utcnow().isoformat() + "Z")
CONFIG_PATH.write_text(json.dumps(RUN_CONFIG, indent=2))

# ---- shard helpers
def atomic_save(obj, path):
    tmp = Path(str(path) + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)

def list_shards():
    return sorted(SHARD_DIR.glob("shard_*.pt"))

def load_done():
    done, next_k = set(), 0
    for p in list_shards():
        next_k = max(next_k, int(p.stem.split("_")[1]) + 1)
        try:
            blob = torch.load(p, map_location="cpu", weights_only=False)
            done.update(int(r) for r in blob["row_idx"])
        except Exception as e:
            print(f"  corrupt shard {p.name} ({e}); deleting, its rows will be redone")
            p.unlink()
    return done, next_k

def make_records(row_ids, questions, scores):
    recs = []
    for i, r in enumerate(row_ids):
        m = meta0.loc[r]
        p_yes, p_no = float(scores["p_yes"][i]), float(scores["p_no"][i])
        pred = "yes" if p_yes > p_no else "no"
        recs.append(dict(
            row_idx=int(r), id=m["id"], question_id=m["question_id"], image_id=m["image_source"], setting=m["category"],
            question=questions[i], gt_answer=m["answer"], pred_answer=pred,
            hallucination=int(pred == "yes" and m["answer"] == "no"),
            p_yes=p_yes, p_no=p_no, p_yes_norm=float(scores["p_yes_norm"][i]), margin=float(scores["margin"][i]),
            binary_entropy=float(scores["binary_entropy"][i]), vocab_entropy=float(scores["vocab_entropy"][i]),
            answer_mass=p_yes + p_no, top1_token=tok.decode([int(scores["top1_id"][i])]),
            split=split_of(m["image_source"]), prompt=make_prompt(questions[i]),
        ))
    return recs

def run_chunk(chunk):
    feats, recs = [], []
    for b in range(0, len(chunk), BATCH_SIZE):
        sub = chunk[b:b + BATCH_SIZE]
        got = [get_sample(r) for r in sub]
        images, questions = [g[0] for g in got], [g[1] for g in got]
        hidden, logits = forward_batch(questions, images)
        feats.append(hidden)
        recs += make_records(sub, questions, score_logits(logits))
    feats = torch.cat(feats)
    assert torch.isfinite(feats).all(), "non-finite hidden states"
    assert feats.abs().max() < 65000 or STORE_DTYPE != "float16", "hidden state magnitude would overflow float16; set STORE_DTYPE='float32' or 'bfloat16'"
    return dict(row_idx=torch.tensor(chunk), features=feats.to(getattr(torch, STORE_DTYPE)), meta=recs)

Selected 9000 questions (full run = 9000).
Resuming a previous run (config matches).


In [9]:
# [GPU]  GPU REQUIRED - RUN IN COLAB - NOT TESTED LOCALLY   (main loop; safe to interrupt and re-run)
from tqdm.auto import tqdm
done, next_k = load_done()
todo = [r for r in SELECTED if r not in done]
print(f"Already done: {len(done & set(SELECTED))} | to do: {len(todo)}")
t0 = time.time()
pbar = tqdm(total=len(todo))
for s in range(0, len(todo), CHECKPOINT_EVERY):
    chunk = todo[s:s + CHECKPOINT_EVERY]
    atomic_save(run_chunk(chunk), SHARD_DIR / f"shard_{next_k:05d}.pt")
    next_k += 1
    pbar.update(len(chunk))
pbar.close()
print(f"Extraction finished in {(time.time() - t0) / 60:.1f} min")

Already done: 0 | to do: 9000


100%|██████████| 9000/9000 [14:44<00:00, 10.18it/s]

Extraction finished in 14.7 min


## Merge shards into the final files

In [10]:
def merge_shards():
    blobs = [torch.load(p, map_location="cpu", weights_only=False) for p in list_shards()]
    sel = set(SELECTED)
    rows, feats, recs, seen = [], [], [], set()
    for b in blobs:
        for j, r in enumerate(b["row_idx"].tolist()):
            if r in seen or r not in sel:
                continue
            seen.add(r)
            rows.append(r); feats.append(b["features"][j]); recs.append(b["meta"][j])
    perm = np.argsort(rows)
    row_idx = torch.tensor([rows[i] for i in perm])
    features = torch.stack([feats[i] for i in perm])
    meta = pd.DataFrame([recs[i] for i in perm])
    return row_idx, features, meta

row_idx, features, meta = merge_shards()
print("merged:", tuple(features.shape), features.dtype, "| metadata rows:", len(meta))
atomic_save(dict(features=features, row_idx=row_idx, token_position="last_prompt_token",
                 note="features[i] corresponds to metadata row with row_idx[i]; shape [N, n_hidden_state_entries, hidden]"),
            OUT / FEATURES_NAME)
try:
    meta.to_parquet(OUT / META_NAME, index=False)
    META_PATH = OUT / META_NAME
except Exception as e:
    META_PATH = OUT / META_NAME.replace(".parquet", ".csv")
    print("parquet unavailable (", e, ") -> writing CSV")
    meta.to_csv(META_PATH, index=False)
RUN_CONFIG.update(status="complete" if len(meta) == len(SELECTED) else "partial", n_rows_saved=int(len(meta)),
                  finished_utc=datetime.datetime.utcnow().isoformat() + "Z", date=str(datetime.date.today()))
CONFIG_PATH.write_text(json.dumps(RUN_CONFIG, indent=2))
print("wrote", OUT / FEATURES_NAME, META_PATH, CONFIG_PATH)

merged: (9000, 29, 1536) torch.float16 | metadata rows: 9000
wrote C:\Users\Toxic\SC4001\SC4001Assignment_2026\results\cached_features\pope_full_features.pt C:\Users\Toxic\SC4001\SC4001Assignment_2026\results\cached_features\pope_metadata.parquet C:\Users\Toxic\SC4001\SC4001Assignment_2026\results\cached_features\pope_run_config.json


## i. Verification (run at the end)

Expected scale: the existing 100-sample file is 8.94 MB (100 x 29 x 1536 x 2 bytes + overhead), so a **full run of 9,000 questions at one token position should be about 0.8 GB** (9,000 x 29 x 1536 x 2 bytes = 0.80 GB) before metadata. Not a few MB, not tens of GB. The cell flags anything far off.

In [11]:
blob = torch.load(OUT / FEATURES_NAME, map_location="cpu", weights_only=True)
F, R = blob["features"], blob["row_idx"].numpy()
M = pd.read_parquet(META_PATH) if str(META_PATH).endswith(".parquet") else pd.read_csv(META_PATH)
print("features:", tuple(F.shape), F.dtype, "| metadata:", M.shape)
assert len(M) == F.shape[0] == len(R) and (M["row_idx"].to_numpy() == R).all(), "features/metadata misaligned"
missing = sorted(set(SELECTED) - set(R.tolist()))
print("rows selected:", len(SELECTED), "| rows saved:", len(R), "| missing:", len(missing), missing[:10])
print("duplicate row_idx:", int(pd.Series(R).duplicated().sum()), "| non-finite values:", int((~torch.isfinite(F.float())).sum()))
print("max |hidden state| per kept entry (head):", F.float().abs().amax(dim=(0, 2))[:5].tolist(), "... overall max", float(F.float().abs().max()))

size_gb = (OUT / FEATURES_NAME).stat().st_size / 1e9
expected_gb = len(R) * F.shape[1] * F.shape[2] * F.element_size() / 1e9
print(f"\nfeature file size: {size_gb:.3f} GB | expected for {len(R)} rows: {expected_gb:.3f} GB | for 9,000 rows ~0.80 GB")
if not (0.9 * expected_gb <= size_gb <= 1.2 * expected_gb):
    print("!!! FLAG: file size is far from expectation, investigate before using the cache")

print("\nprediction distribution:", M["pred_answer"].value_counts().to_dict())
print("answer_mass: mean %.3f | share < 0.9: %.3f" % (M["answer_mass"].mean(), (M["answer_mass"] < 0.9).mean()))
print("top-1 token counts (head):", M["top1_token"].value_counts().head(6).to_dict())
M["correct"] = (M["pred_answer"] == M["gt_answer"]).astype(int)
g = M.groupby("setting")
summary = pd.DataFrame({
    "n": g.size(),
    "accuracy": g["correct"].mean(),
    "yes_rate": g["pred_answer"].apply(lambda s: (s == "yes").mean()),
    "false_yes_rate (pred yes | gt no)": M[M.gt_answer == "no"].groupby("setting")["hallucination"].mean(),
    "hallucination mean over all rows": g["hallucination"].mean(),
}).round(4)
print("\n", summary)
print("\naccuracy overall: %.4f | false-yes rate overall: %.4f" % (M["correct"].mean(), M[M.gt_answer == "no"]["hallucination"].mean()))
print("split sizes (rows):", M["split"].value_counts().to_dict(), "| images per split:", M.groupby("split")["image_id"].nunique().to_dict())
assert M.groupby("image_id")["split"].nunique().max() == 1, "an image appears in two splits"

features: (9000, 29, 1536) torch.float16 | metadata: (9000, 19)
rows selected: 9000 | rows saved: 9000 | missing: 0 []
duplicate row_idx: 0 | non-finite values: 0
max |hidden state| per kept entry (head): [0.10546875, 5.25, 6.75, 10.625, 11.0625] ... overall max 133.0

feature file size: 0.802 GB | expected for 9000 rows: 0.802 GB | for 9,000 rows ~0.80 GB

prediction distribution: {'no': 5269, 'yes': 3731}
answer_mass: mean 0.997 | share < 0.9: 0.000
top-1 token counts (head): {'No': 4783, 'Yes': 3643, 'no': 509, 'yes': 65}

                 n  accuracy  yes_rate  false_yes_rate (pred yes | gt no)  \
setting                                                                    
adversarial  3000    0.8660    0.4280                              0.062   
popular      3000    0.8803    0.4123                              0.032   
random       3000    0.8893    0.4033                              0.014   

             hallucination mean over all rows  
setting                               

## Download fallback (optional)
Preferred route is Google Drive (`USE_DRIVE = True`). If you did not use Drive, set `DOWNLOAD_AT_END = True` in the config and re-run this cell.

In [ ]:
if DOWNLOAD_AT_END:
    from google.colab import files
    for f in [CONFIG_PATH, META_PATH, OUT / FEATURES_NAME]:
        files.download(str(f))
if not KEEP_SHARDS:
    shutil.rmtree(SHARD_DIR, ignore_errors=True)
print("Done. Copy", FEATURES_NAME, META_PATH.name, CONFIG_NAME, "into the repo's results/cached_features/ (do NOT commit the .pt).")